# Notebook — CNN building blocks, from scratch

Deep-learning frameworks hide two things behind one-line calls: **convolution as
a feature detector**, and **gradient descent via backprop**. This notebook makes
both concrete in plain NumPy, so the PyTorch you write later is not magic.

- **Part A — convolution detects features.** Slide a small filter over an image;
  edges/blur fall out. A ReLU keeps the "this feature is present" signal.
- **Part B — a tiny network learns, with hand-written backprop.** Train a
  1-hidden-layer net on a non-linearly-separable dataset, computing every
  gradient by hand, and watch the loss fall and a curved decision boundary form.
- **Part C — why nonlinearity matters.** A linear model can't separate the data;
  the hidden layer can.

NumPy + Matplotlib only. For real CNNs (a classifier, a U-Net) use the PyTorch /
MONAI tutorials cited in the theory — those run in the `mic_soc` environment.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (10, 4)
print('numpy', np.__version__)

## Part A — a convolution layer is a feature detector

A conv layer slides a small **filter** over the image and records how strongly
each patch matches it. Hand-picked filters already detect edges; a CNN simply
**learns** the filters that are useful for the task. A **ReLU** then keeps the
positive "feature present" responses and zeros the rest.

In [ ]:
def conv2d_valid(I, k):
    """Cross-correlation (what DL calls 'convolution'): slide k over I."""
    kh, kw = k.shape
    H, W = I.shape[0] - kh + 1, I.shape[1] - kw + 1
    out = np.zeros((H, W))
    for i in range(H):
        for j in range(W):
            out[i, j] = np.sum(I[i:i+kh, j:j+kw] * k)
    return out

def relu(x):
    return np.maximum(0.0, x)

# a small synthetic image: a square + a disk on a grey background
N = 64
img = np.full((N, N), 0.2)
img[12:30, 12:50] = 0.8
yy, xx = np.mgrid[0:N, 0:N]
img[(xx-44)**2 + (yy-44)**2 < 11**2] = 0.6

sobel_x = np.array([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=float)
sobel_y = sobel_x.T
blur    = np.ones((3, 3)) / 9.0

fig, ax = plt.subplots(1, 4, figsize=(15, 4))
ax[0].imshow(img, cmap='gray'); ax[0].set_title('input image'); ax[0].axis('off')
ax[1].imshow(relu(conv2d_valid(img, sobel_x)), cmap='gray'); ax[1].set_title('ReLU(Sobel-x): vertical edges'); ax[1].axis('off')
ax[2].imshow(relu(conv2d_valid(img, sobel_y)), cmap='gray'); ax[2].set_title('ReLU(Sobel-y): horizontal edges'); ax[2].axis('off')
ax[3].imshow(conv2d_valid(img, blur), cmap='gray'); ax[3].set_title('blur filter'); ax[3].axis('off')
plt.tight_layout(); plt.show()

print('one 3x3 filter has 9 weights — and detects its feature ANYWHERE in the image (weight sharing).')

## Part B — a tiny network that learns, with hand-written backprop

We train a 1-hidden-layer net `input(2) → tanh(H) → sigmoid(1)` on a
non-linearly-separable "two moons" dataset, using **binary cross-entropy** and
**gradient descent**. Every gradient is written out by hand — this is exactly
what PyTorch's autograd computes for you automatically.

In [ ]:
def make_moons(n=400, noise=0.12):
    n1 = n // 2; n2 = n - n1
    t1 = np.linspace(0, np.pi, n1)
    m1 = np.stack([np.cos(t1), np.sin(t1)], axis=1)
    t2 = np.linspace(0, np.pi, n2)
    m2 = np.stack([1 - np.cos(t2), 0.5 - np.sin(t2)], axis=1)
    X = np.vstack([m1, m2]) + noise * rng.standard_normal((n, 2))
    y = np.concatenate([np.zeros(n1), np.ones(n2)])
    return X, y

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

X, y = make_moons()

# --- parameters of a 2 -> H -> 1 net ---
H = 16
W1 = rng.normal(0, 0.5, (2, H)); b1 = np.zeros(H)
W2 = rng.normal(0, 0.5, (H, 1)); b2 = np.zeros(1)
lr, losses = 0.5, []

for epoch in range(3000):
    # forward
    Z1 = X @ W1 + b1
    A1 = np.tanh(Z1)
    P  = sigmoid(A1 @ W2 + b2)[:, 0]
    loss = -np.mean(y*np.log(P+1e-7) + (1-y)*np.log(1-P+1e-7))
    losses.append(loss)
    # backward (chain rule, by hand)
    dZ2 = ((P - y) / len(y))[:, None]    # d loss / d (pre-sigmoid)
    dW2 = A1.T @ dZ2
    db2 = dZ2.sum(0)
    dZ1 = (dZ2 @ W2.T) * (1 - A1**2)     # times tanh'
    dW1 = X.T @ dZ1
    db1 = dZ1.sum(0)
    # gradient-descent update
    W1 -= lr*dW1; b1 -= lr*db1; W2 -= lr*dW2; b2 -= lr*db2

acc = np.mean((P > 0.5) == y)
print(f'final loss {losses[-1]:.3f}   train accuracy {acc:.3f}')
plt.figure(figsize=(7, 4))
plt.plot(losses); plt.xlabel('epoch'); plt.ylabel('cross-entropy loss')
plt.title('training loss (gradient descent + hand-written backprop)'); plt.show()

In [ ]:
# Part C: the nonlinear net vs a purely linear model (logistic regression)
def mlp_predict(Xq):
    return sigmoid(np.tanh(Xq @ W1 + b1) @ W2 + b2)[:, 0]

# train a linear model (no hidden layer) for comparison
wlin = rng.normal(0, 0.5, 2); blin = 0.0
for _ in range(3000):
    p = sigmoid(X @ wlin + blin)
    g = (p - y) / len(y)
    wlin -= 0.5 * (X.T @ g); blin -= 0.5 * g.sum()
def lin_predict(Xq):
    return sigmoid(Xq @ wlin + blin)

gx, gy = np.meshgrid(np.linspace(X[:,0].min()-0.5, X[:,0].max()+0.5, 200),
                     np.linspace(X[:,1].min()-0.5, X[:,1].max()+0.5, 200))
grid = np.stack([gx.ravel(), gy.ravel()], axis=1)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))
for a_, pred, title in [(ax[0], lin_predict, 'linear model — cannot separate moons'),
                        (ax[1], mlp_predict, 'hidden layer — curved boundary works')]:
    zz = pred(grid).reshape(gx.shape)
    a_.contourf(gx, gy, zz, levels=20, cmap='RdBu', alpha=0.6)
    a_.scatter(X[y==0,0], X[y==0,1], s=10, c='blue', edgecolors='k', linewidths=0.2)
    a_.scatter(X[y==1,0], X[y==1,1], s=10, c='red',  edgecolors='k', linewidths=0.2)
    a_.set_title(title); a_.set_aspect('equal')
plt.tight_layout(); plt.show()

## Exercises

1. Change the hidden width `H` (2, 4, 8, 64) and the noise level; when does the
   boundary under/over-fit?
2. Swap `tanh` for ReLU in the hidden layer (and its derivative `1·(Z1>0)`);
   compare training.
3. Add a second hidden layer (extend the forward and backward passes by hand) —
   you've now implemented a 2-hidden-layer MLP from scratch.
4. In Part A, stack two conv+ReLU layers and show the receptive field grows.
5. Verify your hand gradients with a finite-difference check on one weight.

## Where this goes (real CNNs)

The same loss + backprop + gradient descent, but with **convolution** layers and
**autograd**, is a CNN. For runnable medical examples — a CNN classifier and a
U-Net — use the PyTorch 60-minute blitz and the MONAI 2D-segmentation tutorial in
the `mic_soc` environment. Your **Project 2** (segmentation, in `week_6`) is the
place to put a U-Net to work.